# LAB 4 - PHAN TICH DU LIEU NANG CAO VOI PANDAS VA SQL

- Thoi gian: 2 tiet (90 phut)
- Chu de: Quan ly khoa hoc truc tuyen EduTrack
- Muc tieu:
  - Su dung Pandas de xu ly, bien doi va phan tich du lieu nhieu bang
  - Ket hop SQL (sqlite3) voi Pandas de tran van du lieu phuc tap
  - Lam quen voi cac ky thuat merge, pivot, resample, apply

Luu y:
- Dataset duoc tao san trong notebook, khong can tai file ngoai.
- Cac o co nhan [TODO] la phan ban phai tu viet code.
- Phan huong dan cho vi du san, doc ky truoc khi lam phan TODO.


## BUOC 0 - KHOI TAO DATASET

Chay cell nay truoc. No se tao 4 DataFrame mo phong he thong EduTrack:
- df_khoahoc: Thong tin khoa hoc
- df_hocvien: Thong tin hoc vien
- df_dangky: Phieu dang ky hoc
- df_tientrinhh: Tien trinh hoc cua tung hoc vien theo tung buoi


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# --- BANG 1: KHOA HOC ---
df_khoahoc = pd.DataFrame({
    'ma_khoa': ['KH001', 'KH002', 'KH003', 'KH004', 'KH005'],
    'ten_khoa': [
        'Python Co Ban', 'Machine Learning', 'Web Django',
        'SQL & Database', 'Data Visualization'
    ],
    'linh_vuc': ['Lap trinh', 'AI', 'Lap trinh', 'Database', 'AI'],
    'hoc_phi': [1500000, 3500000, 2000000, 1800000, 2500000],
    'so_buoi': [12, 20, 15, 10, 12],
    'ngay_khai_giang': pd.to_datetime([
        '2024-01-10', '2024-02-01', '2024-01-20',
        '2024-03-05', '2024-02-15'
    ])
})

# --- BANG 2: HOC VIEN ---
ho = ['Nguyen', 'Tran', 'Le', 'Pham', 'Hoang', 'Bui', 'Do', 'Ngo', 'Duong', 'Ly']
ten = ['An', 'Binh', 'Cuong', 'Dung', 'Em', 'Phong', 'Giang', 'Hai', 'Lan', 'Mai']
tinh = ['Ha Noi', 'TP HCM', 'Da Nang', 'Can Tho', 'Hue']

np.random.seed(42)
n_hv = 30
df_hocvien = pd.DataFrame({
    'ma_hv': [f'HV{str(i).zfill(3)}' for i in range(1, n_hv + 1)],
    'ho_ten': [f"{np.random.choice(ho)} {np.random.choice(ten)}" for _ in range(n_hv)],
    'tinh_thanh': np.random.choice(tinh, n_hv),
    'nam_sinh': np.random.randint(1995, 2005, n_hv),
    'ngay_dang_ky_he_thong': pd.to_datetime(
        np.random.choice(pd.date_range('2023-06-01', '2024-01-01'), n_hv)
    )
})

# --- BANG 3: DANG KY KHOA HOC ---
records = []
ma_phieu = 1
for i, row in df_hocvien.iterrows():
    n_dk = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
    khoas = np.random.choice(df_khoahoc['ma_khoa'], n_dk, replace=False)
    for k in khoas:
        records.append({
            'ma_phieu': f'PH{str(ma_phieu).zfill(4)}',
            'ma_hv': row['ma_hv'],
            'ma_khoa': k,
            'ngay_dang_ky': pd.Timestamp('2024-01-01') + pd.Timedelta(days=int(np.random.randint(0, 90))),
            'trang_thai': np.random.choice(['Dang hoc', 'Hoan thanh', 'Bo giang giua chung'], p=[0.4, 0.5, 0.1])
        })
        ma_phieu += 1

df_dangky = pd.DataFrame(records)

# --- BANG 4: TIEN TRINH HOC ---
tien_trinh_records = []
for _, dk in df_dangky.iterrows():
    so_buoi = df_khoahoc.loc[df_khoahoc['ma_khoa'] == dk['ma_khoa'], 'so_buoi'].values[0]
    if dk['trang_thai'] == 'Hoan thanh':
        so_buoi_da_hoc = so_buoi
    elif dk['trang_thai'] == 'Bo giang giua chung':
        so_buoi_da_hoc = int(np.random.randint(1, so_buoi // 2))
    else:
        so_buoi_da_hoc = int(np.random.randint(so_buoi // 2, so_buoi))
    
    for buoi in range(1, so_buoi_da_hoc + 1):
        tien_trinh_records.append({
            'ma_phieu': dk['ma_phieu'],
            'buoi_so': buoi,
            'ngay_hoc': dk['ngay_dang_ky'] + pd.Timedelta(days=buoi * 3),
            'diem_bai_tap': round(np.random.uniform(4.0, 10.0), 1),
            'tham_du': np.random.choice([True, False], p=[0.85, 0.15])
        })

df_tientrinh = pd.DataFrame(tien_trinh_records)

print("Khoi tao du lieu thanh cong!")
print(f"  - Khoa hoc : {len(df_khoahoc)} ban ghi")
print(f"  - Hoc vien : {len(df_hocvien)} ban ghi")
print(f"  - Dang ky  : {len(df_dangky)} ban ghi")
print(f"  - Tien trinh: {len(df_tientrinh)} ban ghi")


---
## PHAN 1 - KHAM PHA DU LIEU

Truoc khi phan tich, can hieu ro cau truc cua tung bang.


### Huong dan 1.1 - Xem tong quan mot bang

Su dung `.info()` de biet kieu du lieu, `.describe()` de biet thong ke co ban, `.head()` de xem mau du lieu.

In [ ]:
# Vi du: kham pha bang dang ky
print("=== THONG TIN BANG DANG KY ===")
df_dangky.info()
print()
df_dangky.head()

In [ ]:
# Vi du: dem so luong trang thai dang ky
print("Phan bo trang thai dang ky:")
print(df_dangky['trang_thai'].value_counts())

### [TODO] 1.2 - Kham pha cac bang con lai

Thuc hien cac yeu cau sau:
1. In ra shape (so hang, so cot) cua ca 4 bang
2. Dung `.head(3)` xem 3 hang dau cua bang `df_hocvien` va `df_tientrinh`
3. In ra danh sach linh_vuc duy nhat trong `df_khoahoc`
4. Phat hien gia tri thieu (neu co) trong `df_dangky` va `df_tientrinh`

In [ ]:
# TODO 1.2.a - In shape cua ca 4 bang
# Goi y: print("Ten bang:", df_xxx.shape)



In [ ]:
# TODO 1.2.b - Xem 3 hang dau cua df_hocvien va df_tientrinh



In [ ]:
# TODO 1.2.c - Liet ke cac linh_vuc duy nhat trong df_khoahoc



In [ ]:
# TODO 1.2.d - Kiem tra gia tri thieu trong df_dangky va df_tientrinh



---
## PHAN 2 - KET NOI BANG VOI PANDAS MERGE

Trong Pandas, `merge()` thuc hien chuc nang tuong tu JOIN trong SQL. Day la ky nang quan trong khi lam viec voi nhieu bang du lieu.

Cac kieu merge:
- `how='inner'`: chi lay hang khop o ca 2 bang (giong INNER JOIN)
- `how='left'`: lay tat ca hang ben trai, NULL neu khong khop (giong LEFT JOIN)
- `how='right'`: lay tat ca hang ben phai
- `how='outer'`: lay tat ca hang ca 2 phia


### Huong dan 2.1 - Merge 2 bang

Ket hop bang dang ky voi thong tin khoa hoc de biet ten khoa, hoc phi.

In [ ]:
# Ket hop dang ky voi khoa hoc qua cot chung 'ma_khoa'
df_dk_khoa = pd.merge(
    df_dangky,
    df_khoahoc[['ma_khoa', 'ten_khoa', 'hoc_phi', 'linh_vuc']],
    on='ma_khoa',
    how='inner'
)
print("So hang sau merge:", len(df_dk_khoa))
df_dk_khoa.head()

### [TODO] 2.2 - Merge 3 bang

Tao bang tong hop `df_tong_hop` chua day du thong tin: ten hoc vien, ten khoa, hoc phi, tinh thanh, trang thai.

Goi y: merge theo thu tu: df_dangky -> df_hocvien -> df_khoahoc

In [ ]:
# TODO 2.2 - Tao df_tong_hop bang cach merge 3 bang
# Buoc 1: merge df_dangky voi df_hocvien tren cot 'ma_hv'
# Buoc 2: merge ket qua tren voi df_khoahoc tren cot 'ma_khoa'
# Chi lay cac cot can thiet: ma_phieu, ho_ten, tinh_thanh, ten_khoa, hoc_phi, linh_vuc, trang_thai, ngay_dang_ky

df_tong_hop = None  # Thay None bang code cua ban

print("Shape:", df_tong_hop.shape)
df_tong_hop.head()

### [TODO] 2.3 - LEFT JOIN de tim hoc vien chua dang ky khoa nao

Su dung `how='left'` tu `df_hocvien` sang `df_dangky`. Nhung hoc vien khong co trong `df_dangky` se co gia tri NaN o cot `ma_phieu`.

In [ ]:
# TODO 2.3
# Merge df_hocvien voi df_dangky, how='left'
# Loc ra nhung hang ma_phieu la NaN
# In ra ho_ten cua nhung hoc vien chua dang ky khoa nao



---
## PHAN 3 - THONG KE VA NHOM DU LIEU (GROUPBY NANG CAO)


### Huong dan 3.1 - GroupBy ket hop nhieu ham thong ke

Dung `.agg()` de tinh nhieu chi so cung luc.

In [ ]:
# Vi du: thong ke theo linh_vuc
thongke_linhvuc = df_tong_hop.groupby('linh_vuc').agg(
    so_luot_dang_ky=('ma_phieu', 'count'),
    tong_doanh_thu=('hoc_phi', 'sum'),
    so_hoc_vien_khac_nhau=('ho_ten', 'nunique')
).reset_index()

thongke_linhvuc['trung_binh_doanh_thu'] = (
    thongke_linhvuc['tong_doanh_thu'] / thongke_linhvuc['so_luot_dang_ky']
).round(0)

print(thongke_linhvuc)

### [TODO] 3.2 - Thong ke theo ten_khoa

Tao bang thong ke `thongke_khoa` tu `df_tong_hop`, nhom theo `ten_khoa`, tinh:
- `so_dang_ky`: tong so phieu dang ky
- `so_hoan_thanh`: dem so phieu co trang_thai == 'Hoan thanh'
- `tong_doanh_thu`: tong hoc_phi thu duoc
- `ty_le_hoan_thanh`: so_hoan_thanh / so_dang_ky * 100, lam tron 1 chu so

Sap xep giam dan theo tong_doanh_thu.

In [ ]:
# TODO 3.2



### [TODO] 3.3 - Thong ke theo tinh_thanh

Tim 3 tinh/thanh pho co tong hoc_phi cao nhat. Chi hien thi 2 cot: `tinh_thanh` va `tong_hoc_phi`.

In [ ]:
# TODO 3.3



---
## PHAN 4 - PIVOT TABLE

Pivot table giup chuyen du lieu tu dang "dai" sang dang "rong", de nhin hoan canh nhieu chieu.


### Huong dan 4.1 - Pivot table so luot dang ky theo tinh va linh vuc

In [ ]:
pivot_tinh_linhvuc = pd.pivot_table(
    df_tong_hop,
    values='ma_phieu',
    index='tinh_thanh',
    columns='linh_vuc',
    aggfunc='count',
    fill_value=0
)
print(pivot_tinh_linhvuc)

### [TODO] 4.2 - Pivot table doanh thu theo tinh va ten_khoa

Tao pivot table voi:
- `index`: tinh_thanh
- `columns`: ten_khoa
- `values`: hoc_phi
- `aggfunc`: sum
- `fill_value`: 0

Sau do them hang `Total` la tong tung cot va cot `Total` la tong tung hang.

In [ ]:
# TODO 4.2
# Goi y them tong: pivot['Total'] = pivot.sum(axis=1)
#                  pivot.loc['Total'] = pivot.sum()



### [TODO] 4.3 - Pivot table ty le hoan thanh theo tinh va linh vuc

Tao pivot table cho biet ty le % hoc vien 'Hoan thanh' theo tung tinh_thanh va linh_vuc.

Goi y: Tao cot `hoan_thanh` = 1 neu trang_thai == 'Hoan thanh', roi dung `aggfunc='mean'` va nhan 100.

In [ ]:
# TODO 4.3



---
## PHAN 5 - PHAN TICH THEO THOI GIAN (TIME SERIES)


### Huong dan 5.1 - Dem so dang ky theo thang

In [ ]:
# Dam bao cot ngay_dang_ky dung kieu datetime
df_tong_hop['ngay_dang_ky'] = pd.to_datetime(df_tong_hop['ngay_dang_ky'])

# Trich xuat thang
df_tong_hop['thang'] = df_tong_hop['ngay_dang_ky'].dt.to_period('M')

# Dem so dang ky theo thang
dk_theo_thang = df_tong_hop.groupby('thang')['ma_phieu'].count()
print("So dang ky theo thang:")
print(dk_theo_thang)

### [TODO] 5.2 - Doanh thu luy ke theo thang

Tu `df_tong_hop`:
1. Tinh tong hoc_phi theo tung thang
2. Tinh doanh thu luy ke (cumsum) va luu vao cot `luy_ke`
3. In ra ket qua

In [ ]:
# TODO 5.2
# Goi y: series.cumsum() tinh gia tri luy ke



### [TODO] 5.3 - Phan tich tien trinh hoc theo tuan

Su dung `df_tientrinh` da merge voi `df_dangky` de lay `ma_hv`:
1. Merge `df_tientrinh` voi `df_dangky` tren cot `ma_phieu` (lay them cot `ma_khoa`)
2. Tinh diem_bai_tap trung binh theo tung `buoi_so`
3. Tim buoi co diem trung binh thap nhat va cao nhat

In [ ]:
# TODO 5.3



---
## PHAN 6 - KET HOP SQL VA PANDAS

Pandas doc du lieu bang SQL thong qua `pd.read_sql_query()`. Day la cach ket hop linh hoat giua 2 cong cu.


### Huong dan 6.1 - Nap du lieu vao SQLite va tran van bang SQL

In [ ]:
# Tao ket noi SQLite in-memory (khong can file)
conn = sqlite3.connect(':memory:')

# Nap 4 bang vao SQLite
df_khoahoc.to_sql('KhoaHoc', conn, index=False, if_exists='replace')
df_hocvien.to_sql('HocVien', conn, index=False, if_exists='replace')
df_dangky.to_sql('DangKy', conn, index=False, if_exists='replace')
df_tientrinh.to_sql('TienTrinh', conn, index=False, if_exists='replace')

print("Da nap xong du lieu vao SQLite!")

# Vi du: tran van bang SQL
sql = """
SELECT k.ten_khoa, COUNT(d.ma_phieu) AS so_dang_ky
FROM DangKy d
JOIN KhoaHoc k ON d.ma_khoa = k.ma_khoa
GROUP BY k.ten_khoa
ORDER BY so_dang_ky DESC
"""
result = pd.read_sql_query(sql, conn)
print(result)

### [TODO] 6.2 - Viet cau lenh SQL tim hoc vien dang hoc nhieu khoa nhat

Viet cau lenh SQL tra ve: `ho_ten`, `so_khoa_dang_hoc` (dem cac phieu co trang_thai = 'Dang hoc'), sap xep giam dan, lay top 5.

In [ ]:
# TODO 6.2
sql_todo = """
-- Viet cau lenh SQL cua ban o day
"""
# pd.read_sql_query(sql_todo, conn)


### [TODO] 6.3 - Viet cau lenh SQL thong ke doanh thu theo linh vuc

Viet SQL JOIN 3 bang (DangKy, KhoaHoc, HocVien), nhom theo `linh_vuc`, tinh:
- `tong_doanh_thu` = SUM(hoc_phi)
- `so_hoc_vien` = COUNT DISTINCT ma_hv

Chi tinh cho nhung phieu co trang_thai khac 'Bo giang giua chung'.

In [ ]:
# TODO 6.3
sql_todo2 = """
-- Viet cau lenh SQL cua ban o day
"""
# pd.read_sql_query(sql_todo2, conn)


### [TODO] 6.4 - Subquery trong SQL

Viet SQL dung Subquery de tim cac khoa hoc co hoc phi cao hon muc hoc phi trung binh cua toan bo khoa hoc.

Tra ve: `ten_khoa`, `hoc_phi`, `linh_vuc`.

In [ ]:
# TODO 6.4
sql_todo3 = """
-- Viet cau lenh SQL cua ban o day
"""
# pd.read_sql_query(sql_todo3, conn)


---
## PHAN 7 - XU LY DU LIEU NANG CAO VOI APPLY

`apply()` cho phep ap dung mot ham tu dinh nghia len tung hang hoac cot cua DataFrame.


### Huong dan 7.1 - Phan loai hoc vien theo so khoa dang ky

In [ ]:
# Dem so khoa moi hoc vien dang ky
so_khoa_moi_hv = df_dangky.groupby('ma_hv')['ma_khoa'].count().reset_index()
so_khoa_moi_hv.columns = ['ma_hv', 'so_khoa']

# Ham phan loai
def phan_loai_hv(so_khoa):
    if so_khoa >= 3:
        return 'Tich cuc'
    elif so_khoa == 2:
        return 'Binh thuong'
    else:
        return 'Thu dong'

so_khoa_moi_hv['muc_do_tham_gia'] = so_khoa_moi_hv['so_khoa'].apply(phan_loai_hv)
print(so_khoa_moi_hv['muc_do_tham_gia'].value_counts())

### [TODO] 7.2 - Phan loai khoa hoc theo muc hoc phi

Viet ham `phan_loai_hoc_phi(hoc_phi)` tra ve:
- 'Cao cap' neu hoc_phi >= 3000000
- 'Trung cap' neu hoc_phi >= 1800000
- 'Pho cap' neu nho hon

Ap dung len cot `hoc_phi` cua `df_khoahoc`, luu vao cot moi `phan_khuc`.

In [ ]:
# TODO 7.2



### [TODO] 7.3 - Tinh diem trung binh cua moi phieu mua

Tu `df_tientrinh`, viet code:
1. Tinh diem_bai_tap trung binh theo tung `ma_phieu`
2. Phan loai ket qua: >= 8.0 la 'Gioi', >= 6.5 la 'Kha', >= 5.0 la 'Trung binh', con lai la 'Yeu'
3. Merge ket qua vao `df_dangky` theo `ma_phieu`
4. In ra phan bo xep loai

In [ ]:
# TODO 7.3



---
## PHAN 8 - XUAT DU LIEU

Sau khi phan tich xong, luu ket qua ra file CSV va Excel.


### Huong dan 8.1 - Luu file CSV

In [ ]:
# Luu bang tong hop ra CSV
df_tong_hop.to_csv('edutrack_tong_hop.csv', index=False, encoding='utf-8-sig')
print('Da luu file edutrack_tong_hop.csv')

### [TODO] 8.2 - Xuat file Excel nhieu sheet

Xuat file `ket_qua_lab4.xlsx` gom 3 sheet:
- Sheet `TongHop`: noi dung df_tong_hop
- Sheet `ThongKe_Khoa`: bang thongke_khoa da tao o Phan 3.2
- Sheet `Pivot_DoanhThu`: pivot table da tao o Phan 4.2

Goi y: Dung `pd.ExcelWriter` voi `with` block.

In [ ]:
# TODO 8.2



---
## TOM TAT LAB 4

Trong lab nay ban da luyen tap:

1. Kham pha du lieu nhieu bang lien ket
2. Ket hop bang voi `pd.merge()` (inner, left join)
3. Thong ke nang cao voi `groupby().agg()`
4. Tao pivot table nhieu chieu
5. Phan tich theo thoi gian va tinh luy ke
6. Ket hop SQL va Pandas qua sqlite3
7. Bien doi du lieu tuy chinh voi `apply()`
8. Xuat du lieu ra CSV va Excel nhieu sheet

Chuan bi cho Lab 5: Su dung toan bo ket qua phan tich de ve bieu do chuyen sau voi Matplotlib va Seaborn.
